## 1、流式调用 、非流式调用

In [1]:
# 非流式调用
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, SystemMessage, HumanMessage
import dotenv
from openai import max_retries
from sympy.physics.units import temperature

dotenv.load_dotenv()
# 1、获取大模型的实例
model = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    temperature=0.5,
)
res = model.invoke("你好，你是谁")
print(res)

C:\Users\m1881\miniconda3\envs\LangChainProj\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


content='你好！我是DeepSeek，由深度求索公司创造的AI助手！😊\n\n我是一个纯文本模型，虽然不支持多模态识别功能，但我可以帮你处理上传的各种文件，比如图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息进行分析处理。我拥有128K的上下文长度，可以处理比较长的对话和文档。\n\n我目前是完全免费的，你可以通过官方应用商店下载App使用我。如果需要联网搜索功能的话，需要你在Web或App上手动点开联网搜索按键。\n\n我很乐意为你提供各种帮助，无论是回答问题、协助工作、学习辅导还是日常聊天，都可以找我！有什么我可以帮你的吗？✨' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 142, 'prompt_tokens': 7, 'total_tokens': 149, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 7}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': '10a720f9-48e0-4d7b-99d7-bd0380f15cb6', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--109a9a8b-856c-4d83-a22c-5a6698ebd1dc-0' usage_metadata={'input_tokens': 7, 'output_tokens': 142, 'total_tokens': 149, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}

In [10]:
# 流式调用，通过model.stream方法去调用，
# 返回的是一个生成器，通过迭代生成器的方式，得到结果
res = model.stream("你好，你是谁")

In [11]:
for chunk in res:
    print(chunk.content, end="")

你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文长度，还支持联网搜索功能（需要你在Web/App中手动开启）。

我很乐意为你提供各种帮助，无论是回答问题、协助工作、学习辅导，还是日常聊天，我都会热情细致地为你服务！有什么我可以帮你的吗？✨

## 2、批次调用、非批次调用

In [12]:
# 批次调用，通过model.batch方法实现，底层原理就是通过多线程的方式去调用，
#
messages = [
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于春天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于夏天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于秋天的诗"},
    ],
]
res = model.batch(messages)

## 3、同步调用、异步调用

In [2]:
# 同步调用：多次请求之间串行处理，B请求需要A请求完成之后，再发出请求，得到响应
messagess = [
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于春天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于夏天的诗"},
    ],
    [
        {"role": "system", "content": "你是一位诗人"},
        {"role": "user", "content": "写一首关于秋天的诗"},
    ],
]
import time

start_time = time.time()
res = [model.invoke(messages) for messages in messagess]
end_time = time.time()
print(f"总耗时:{end_time - start_time}")

总耗时:18.973124027252197


In [3]:
# 异步调用：model.ainvoke方法，返回一个协程对象，把多个协程对象可以打包成一个协程对象，
# await最终的协程对象，就能够实现异步调用
# 异步调用，能够提高程序的性能。相对于batch调用而言，能够减少资源（线程数）使用量
import asyncio


async def gather_task(messages: list):
    # 调用ainvoke并不会真正地发起请求
    tasks = [model.ainvoke(message_list) for message_list in messages]
    return await asyncio.gather(*tasks)


gather_task(messagess)

<coroutine object gather_task at 0x000002C3B7730D60>

In [6]:
await gather_task(messagess)
# asyncio.run(gather_task(messagess))

[AIMessage(content='《春日来信》\n\n东风拆开冰的封印\n泥土翻身 抖落霜的斗篷\n草芽用绿色针脚\n把大地的裂痕细细缝拢\n\n桃枝在窗前写信\n蘸着雨水 写一页粉红\n待燕子剪开雾幔\n将芬芳投递至每一扇窗栊\n\n柳絮是飘浮的邮戳\n盖满三月的天空\n当流云读过所有绽放\n便化作淅沥的批注 落进晚钟\n\n蝴蝶驮着光舞蹈\n在花蕊里酿造甜美的梦\n春天啊 这封长信\n永远寄不到秋天的怀中', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 136, 'prompt_tokens': 12, 'total_tokens': 148, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 12}, 'model_provider': 'deepseek', 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_ffc7281d48_prod0820_fp8_kvcache', 'id': '30eed50f-f608-4bdb-af44-06ace81da89d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--a6431d34-ea5b-4f69-87dc-d94367b9aa14-0', usage_metadata={'input_tokens': 12, 'output_tokens': 136, 'total_tokens': 148, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}),
 AIMessage(content='《夏日的密语》\n\n蝉鸣织成细密的网\n兜住整个正午的河流\n荷叶在风中翻动钱币\n购买一